# 02 · Junction agents and masters
Every signalised junction runs **five agents** and one **master**:

| Agent | Asks for |
|---|---|
| Traffic light | the next green phase (max-pressure) |
| Pedestrian | a walk signal after a long wait; holds the green while someone crosses |
| Transit priority | priority for **late** buses only |
| Emergency priority | green for ambulances, police cars, fire trucks |
| Sensor health | marks frozen detectors so the junction falls back to safer control |

Agents only **ask**. The master decides in a fixed order (safety > emergency > supermaster directives >
transit > pedestrians > normal) and is the only code that writes its light. Here we run 15 minutes of
the **emergency** scenario and look inside.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

## 1. Run a short scenario and capture the junctions every minute

In [ ]:
def capture(now, system):
    row = {"t": now}
    for m in system.masters:
        st = system.bus.status.get(m.tls, {})
        inbox = system.bus.inbox(m.tls, now)
        row[m.tls] = f"{st.get('mode', '?')}, queue {st.get('queue_total', '?')}, {len(inbox)} msg"
    return row
result, system, snaps, narr = nt.short_run("emergency", minutes=15, every=60, capture=capture)
print(f"Car wait {result['veh_mean_wait_s']:.1f} s · pedestrian wait {result['ped_mean_wait_s']:.1f} s · "
      f"truncations {result['clearance_truncations']} · emergency trip {result.get('ambulance_mean_travel_s', float('nan')):.0f} s")
pd.DataFrame(snaps).set_index("t").head(8)

## 2. The agents inside one junction

In [ ]:
m = next(x for x in system.masters if x.tls == "B3")
pd.DataFrame([{"agent": type(a).__name__, "file": type(a).__module__.replace(".", "/") + ".py"} for a in m.agents])

## 3. Who decided what
Every decision is logged with a reason code. Counting them shows the agents at work:

In [ ]:
log = pd.DataFrame(system.log.entries)
log["agent type"] = log.agent.str.split("@").str[0]
log.groupby(["agent type", "reason"]).size().rename("decisions").reset_index().sort_values("decisions", ascending=False).head(20)

## 4. An ambulance, step by step
The emergency priority agent requests green, the supermaster's emergency route agent warns junctions
ahead, the master grants (after checking the exit road), then normal control resumes.

In [ ]:
amb = [e for e in system.log.entries if str(e["details"].get("vehicle", "")).startswith("ambulance")]
first = amb[0]["details"]["vehicle"]
print("Following", first)
timeline = [{"time": narr.clock(e["t"]), "agent": e["agent"], "what happened": narr.sentence(e)}
            for e in system.log.entries if e["details"].get("vehicle") == first]
pd.DataFrame(timeline).head(12)

## 5. Pedestrians and late buses

In [ ]:
display(nt.decisions(system, narr, reasons=["PED_", "CLEARANCE"], limit=6))
display(nt.decisions(system, narr, reasons=["TRANSIT", "BUS_"], limit=6))

## 6. The same decision in three languages
The narrator turns each log entry into a sentence in English, French or Arabic, without a language model.

In [ ]:
e = next(x for x in system.log.entries if x["reason"].startswith("EMERGENCY_GRANTED"))
for lang in ("en", "fr", "ar"):
    print(lang, "·", narr.sentence(e, lang))